# PDF → Pandas DataFrame
### School of Dandori Course Extractor

This notebook breaks extraction into discrete, testable stages. Run each cell individually to verify the output before moving on.

---
## Stage 0 · Setup

Install dependencies and set the path to your PDF folder.

In [1]:
# Install required libraries (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pdfplumber", "pandas", "--quiet"])
print("Dependencies ready.")

Dependencies ready.


In [ ]:
import re, glob
from pathlib import Path
import pdfplumber
import pandas as pd

# ── CONFIGURE THIS ──────────────────────────────────────────────────────────
PDF_DIR = "./pdf_samples/"          # folder containing your PDFs (use "." for current dir)
# ───────────────────────────────────────────────────────────────────────────

pdf_paths = sorted(glob.glob(f"{PDF_DIR}/*.pdf"))
print(f"Found {len(pdf_paths)} PDF(s):")
for p in pdf_paths:
    print(f"  {p}")

Found 4 PDF(s):
  ./pdf-samples\class_001_The_Art_of_Wondrous_Waffle_Weaving.pdf
  ./pdf-samples\class_008_Magical_Woolly_Mammoth_Sculpting.pdf
  ./pdf-samples\class_015_the_magical_marmalade_adventure.pdf
  ./pdf-samples\class_023_the_art_of_whimsical_waffle_weaving.pdf


---
## Stage 1 · Raw Text Extraction

Open each PDF with `pdfplumber` and extract plain text page by page.

Run this to confirm the library can read your files and see what the raw text looks like.

In [3]:
# Pick one PDF to inspect — change the index to try different files
SAMPLE_IDX = 0
sample_path = pdf_paths[SAMPLE_IDX]

pages_text = []
with pdfplumber.open(sample_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        pages_text.append(text)
        print(f"--- Page {i+1} ({len(text)} chars) ---")
        print(text[:600])
        print()

--- Page 1 (694 chars) ---
The Art of Wondrous Waffle Weaving
Instructor: Location:
Chef Waffleby Harrogate
Course Type: Cost:
Culinary Arts £75.00
Learning Objectives
• Master the technique of creating intricate waffle patterns.
• Understand the science behind perfect waffle batter consistency.
• Explore innovative flavour combinations for both sweet and savoury waffles.
• Learn artistic presentation techniques for waffle dishes.
• Develop skills in using waffle irons of various designs.
Provided Materials
• Professional waffle iron
• Selection of flours (including gluten-free options)
• Assorted spices and herbs
• Fre

--- Page 2 (1744 chars) ---
Skills Developed
Culinary Arts Baking Creative Cooking Food Presentation
Flavour Pairing
Course Description
Join us at the Harrogate Culinary Academy for an unforgettable experience in 'The
Art of Wondrous Waffle Weaving.' Under the whimsical guidance of Chef Waffleby,
an aficionado of both traditional and contemporary waffle arts, students 

---
## Stage 2 · Extract Simple Fields

Parse the scalar fields from page 1: **Title**, **Class ID**, **Instructor**, **Location**, **Course Type**, and **Cost**.

In [4]:
p1 = pages_text[0]
full_text = "\n".join(pages_text)

# Title — first non-empty line of page 1
p1_lines = [l.strip() for l in p1.splitlines() if l.strip()]
title = p1_lines[0]
print(f"Title:       {title!r}")

# Class ID
m = re.search(r"Class ID:\s*(CLASS_\w+)", full_text)
class_id = m.group(1) if m else None
print(f"Class ID:    {class_id!r}")

# Instructor & Location
# Layout on page 1: "Instructor: Location:\n<name> <city>"
# Location is always one word, so rsplit on last space.
m = re.search(r"Instructor:\s+Location:\s*\n(.+)", p1)
if m:
    val = m.group(1).strip()
    parts = val.rsplit(" ", 1)
    instructor = parts[0].strip()
    location   = parts[1].strip() if len(parts) > 1 else None
else:
    instructor = location = None
print(f"Instructor:  {instructor!r}")
print(f"Location:    {location!r}")

# Course Type & Cost
# Layout: "Course Type: Cost:\n<type> £<amount>"
m = re.search(r"Course Type:\s+Cost:\s*\n(.+)", p1)
if m:
    val = m.group(1).strip()
    cost_m = re.search(r"(£[\d,.]+)", val)
    if cost_m:
        course_type = val[:val.index(cost_m.group(1))].strip()
        cost_gbp    = float(re.sub(r"[£,]", "", cost_m.group(1)))
    else:
        course_type = val
        cost_gbp    = None
else:
    course_type = cost_gbp = None
print(f"Course Type: {course_type!r}")
print(f"Cost (£):    {cost_gbp}")

Title:       'The Art of Wondrous Waffle Weaving'
Class ID:    'CLASS_4033'
Instructor:  'Chef Waffleby'
Location:    'Harrogate'
Course Type: 'Culinary Arts'
Cost (£):    75.0


---
## Stage 3 · Extract Bullet Lists

Parse **Learning Objectives** and **Provided Materials** by finding bullet `•` characters under each section header.

In [5]:
def extract_bullet_section(text, header):
    """Return a list of strings for each bullet under `header`."""
    pattern = rf"{re.escape(header)}\s*\n(.*?)(?=\n[A-Z][^\n]{{2,}}\n|\Z)"
    m = re.search(pattern, text, re.DOTALL)
    if not m:
        return []
    return [i.strip() for i in re.findall(r"[•\-\*]\s*(.+)", m.group(1))]

learning_objectives = extract_bullet_section(p1, "Learning Objectives")
provided_materials  = extract_bullet_section(p1, "Provided Materials")

print("Learning Objectives:")
for obj in learning_objectives:
    print(f"  • {obj}")

print("\nProvided Materials:")
for mat in provided_materials:
    print(f"  • {mat}")

Learning Objectives:
  • Master the technique of creating intricate waffle patterns.
  • Understand the science behind perfect waffle batter consistency.
  • Explore innovative flavour combinations for both sweet and savoury waffles.
  • Learn artistic presentation techniques for waffle dishes.
  • Develop skills in using waffle irons of various designs.

Provided Materials:
  • Professional waffle iron
  • Selection of flours (including gluten-free options)
  • Assorted spices and herbs
  • Fresh fruits and savory toppings
  • Waffle weaving toolkit (spatulas, piping bags, edible glitter)


---
## Stage 4 · Extract Skills (Positional Parsing)

Skills are rendered as spaced tags on page 2. Plain text collapses the spacing, so we use `pdfplumber`'s **word bounding boxes** to detect gaps between tags and split them correctly.

In [6]:
def extract_skills(page):
    """
    Parse 'Skills Developed' tags using word x-positions.
    Words separated by a gap > 20 pts belong to different skill tags.
    """
    words = page.extract_words()
    sorted_words = sorted(words, key=lambda w: (w["top"], w["x0"]))

    # Find the 'Skills Developed' header (Skills + Developed on same line)
    header_bottom = None
    for i, w in enumerate(sorted_words):
        if w["text"] == "Skills" and i + 1 < len(sorted_words):
            nxt = sorted_words[i + 1]
            if nxt["text"] == "Developed" and abs(nxt["top"] - w["top"]) < 4:
                header_bottom = w["bottom"]
                break

    if header_bottom is None:
        return []

    # Find 'Course Description' header to know where skills section ends
    section_top = 9999
    for i, w in enumerate(sorted_words):
        if w["text"] == "Course" and i + 1 < len(sorted_words):
            nxt = sorted_words[i + 1]
            if nxt["text"] == "Description" and abs(nxt["top"] - w["top"]) < 4:
                section_top = w["top"]
                break

    # Collect words between the two headers
    skill_words = [
        w for w in sorted_words
        if w["top"] >= header_bottom and w["bottom"] <= section_top
    ]
    if not skill_words:
        return []

    # Group into lines (same vertical position ± 4 pts)
    lines, current_line = [], [skill_words[0]]
    for w in skill_words[1:]:
        if abs(w["top"] - current_line[-1]["top"]) < 4:
            current_line.append(w)
        else:
            lines.append(current_line)
            current_line = [w]
    lines.append(current_line)

    # Within each line, split on horizontal gaps > 20 pts
    skills = []
    for line in lines:
        groups, current_group = [], [line[0]]
        for w in line[1:]:
            if w["x0"] - current_group[-1]["x1"] > 20:
                groups.append(current_group)
                current_group = [w]
            else:
                current_group.append(w)
        groups.append(current_group)
        for group in groups:
            tag = " ".join(g["text"] for g in group).strip()
            if tag:
                skills.append(tag)

    return skills


# Test on the sample PDF
with pdfplumber.open(sample_path) as pdf:
    p2_page = pdf.pages[1] if len(pdf.pages) > 1 else None
    skills = extract_skills(p2_page) if p2_page else []

print("Skills Developed:")
for s in skills:
    print(f"  · {s}")

Skills Developed:
  · Culinary Arts
  · Baking
  · Creative Cooking
  · Food Presentation
  · Flavour Pairing


---
## Stage 5 · Extract Description

Pull the full course description text from page 2, collapsing soft line-breaks.

In [7]:
m = re.search(r"Course Description\s*\n(.*?)(?=Class ID:|\Z)", full_text, re.DOTALL)
if m:
    desc = re.sub(r"(?<!\n)\n(?!\n)", " ", m.group(1).strip())
    description = desc.strip()
else:
    description = None

print(description)

Join us at the Harrogate Culinary Academy for an unforgettable experience in 'The Art of Wondrous Waffle Weaving.' Under the whimsical guidance of Chef Waffleby, an aficionado of both traditional and contemporary waffle arts, students will embark on a culinary journey like no other. In this delightful class, you'll learn how to transform the humble waffle into an edible masterpiece through the innovative practice of waffle weaving. The class begins with a foundation in waffle batter alchemy, where you'll discover the delicate balance of ingredients needed to achieve the perfect texture. Once you've mastered the basics, Chef Waffleby will introduce you to the world of waffle pattern creation, teaching you how to weave intricate designs that are as pleasing to the eye as they are to the palate. Students will experiment with a spectrum of flavours, crafting sweet creations with fruits and edible flowers and savory delights with herbs and spices. You'll also learn to enhance your waffle pr

---
## Stage 6 · Single-PDF Record

Combine all stages into one dictionary for the sample PDF. This is what one row of the final DataFrame will look like.

In [8]:
record = {
    "file_name":            Path(sample_path).name,
    "class_id":             class_id,
    "title":                title,
    "instructor":           instructor,
    "location":             location,
    "course_type":          course_type,
    "cost_gbp":             cost_gbp,
    "learning_objectives":  learning_objectives,
    "provided_materials":   provided_materials,
    "skills_developed":     skills,
    "description":          description,
}

for k, v in record.items():
    if isinstance(v, list):
        print(f"{k}:")
        for item in v:
            print(f"    {item}")
    else:
        print(f"{k}: {v}")

file_name: class_001_The_Art_of_Wondrous_Waffle_Weaving.pdf
class_id: CLASS_4033
title: The Art of Wondrous Waffle Weaving
instructor: Chef Waffleby
location: Harrogate
course_type: Culinary Arts
cost_gbp: 75.0
learning_objectives:
    Master the technique of creating intricate waffle patterns.
    Understand the science behind perfect waffle batter consistency.
    Explore innovative flavour combinations for both sweet and savoury waffles.
    Learn artistic presentation techniques for waffle dishes.
    Develop skills in using waffle irons of various designs.
provided_materials:
    Professional waffle iron
    Selection of flours (including gluten-free options)
    Assorted spices and herbs
    Fresh fruits and savory toppings
    Waffle weaving toolkit (spatulas, piping bags, edible glitter)
skills_developed:
    Culinary Arts
    Baking
    Creative Cooking
    Food Presentation
    Flavour Pairing
description: Join us at the Harrogate Culinary Academy for an unforgettable experie

---
## Stage 7 · Wrap into Functions

Package the logic from stages 1–5 into reusable functions.

In [9]:
def extract_course_data(pdf_path: str) -> dict:
    """Extract all fields from a single School of Dandori PDF."""
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            pages_text.append(page.extract_text() or "")
        page2_skills = extract_skills(pdf.pages[1]) if len(pdf.pages) > 1 else []

    full_text = "\n".join(pages_text)
    p1 = pages_text[0]
    p1_lines = [l.strip() for l in p1.splitlines() if l.strip()]

    # Title
    title = p1_lines[0] if p1_lines else None

    # Class ID
    m = re.search(r"Class ID:\s*(CLASS_\w+)", full_text)
    class_id = m.group(1) if m else None

    # Instructor & Location
    m = re.search(r"Instructor:\s+Location:\s*\n(.+)", p1)
    if m:
        parts = m.group(1).strip().rsplit(" ", 1)
        instructor = parts[0].strip()
        location   = parts[1].strip() if len(parts) > 1 else None
    else:
        instructor = location = None

    # Course Type & Cost
    m = re.search(r"Course Type:\s+Cost:\s*\n(.+)", p1)
    if m:
        val    = m.group(1).strip()
        cost_m = re.search(r"(£[\d,.]+)", val)
        course_type = val[:val.index(cost_m.group(1))].strip() if cost_m else val
        cost_gbp    = float(re.sub(r"[£,]", "", cost_m.group(1))) if cost_m else None
    else:
        course_type = cost_gbp = None

    # Bullet lists
    learning_objectives = extract_bullet_section(p1, "Learning Objectives")
    provided_materials  = extract_bullet_section(p1, "Provided Materials")

    # Description
    m = re.search(r"Course Description\s*\n(.*?)(?=Class ID:|\Z)", full_text, re.DOTALL)
    description = re.sub(r"(?<!\n)\n(?!\n)", " ", m.group(1).strip()).strip() if m else None

    return {
        "file_name":           Path(pdf_path).name,
        "class_id":            class_id,
        "title":               title,
        "instructor":          instructor,
        "location":            location,
        "course_type":         course_type,
        "cost_gbp":            cost_gbp,
        "learning_objectives": learning_objectives,
        "provided_materials":  provided_materials,
        "skills_developed":    page2_skills,
        "description":         description,
    }


def pdfs_to_dataframe(pdf_paths: list) -> pd.DataFrame:
    """Parse a list of PDF paths and return a DataFrame (one row per PDF)."""
    records = []
    for path in pdf_paths:
        try:
            records.append(extract_course_data(path))
            print(f"  ✓  {Path(path).name}")
        except Exception as e:
            print(f"  ✗  {Path(path).name}  →  {e}")
    return pd.DataFrame(records)

print("Functions defined: extract_bullet_section, extract_skills, extract_course_data, pdfs_to_dataframe")

Functions defined: extract_bullet_section, extract_skills, extract_course_data, pdfs_to_dataframe


---
## Stage 8 · Build the Full DataFrame

Run the extractor across all PDFs and display the results.

In [10]:
print(f"Processing {len(pdf_paths)} PDF(s)...\n")
df = pdfs_to_dataframe(pdf_paths)

print(f"\nDataFrame shape: {df.shape}")
df

Processing 4 PDF(s)...

  ✓  class_001_The_Art_of_Wondrous_Waffle_Weaving.pdf
  ✓  class_008_Magical_Woolly_Mammoth_Sculpting.pdf
  ✓  class_015_the_magical_marmalade_adventure.pdf
  ✓  class_023_the_art_of_whimsical_waffle_weaving.pdf

DataFrame shape: (4, 11)


,file_name,class_id,title,instructor,location,course_type,cost_gbp,learning_objectives,provided_materials,skills_developed,description
0,class_001_The_Art_of_Wondrous_Waffle_Weaving.pdf,CLASS_4033,The Art of Wondrous Waffle Weaving,Chef Waffleby,Harrogate,Culinary Arts,75.0,[Master the technique of creating intricate wa...,"[Professional waffle iron, Selection of flours...","[Culinary Arts, Baking, Creative Cooking, Food...",Join us at the Harrogate Culinary Academy for ...
1,class_008_Magical_Woolly_Mammoth_Sculpting.pdf,CLASS_8046,Magical Woolly Mammoth Sculpting,Professor Woolly Mammoth,Northumberland,Fiber Arts,75.0,[Master the basics of needle felting to create...,"[Selection of colourful wool, Felting needles,...","[Needle Felting, Sculpting, Artistic Expressio...",Step into the whimsical world of 'Magical Wool...
2,class_015_the_magical_marmalade_adventure.pdf,CLASS_0425,The Magical Marmalade Adventure,Chef Marmalock,Inverness,Culinary Arts,55.0,[Master the art of making traditional Scottish...,"[Citrus fruits (oranges, lemons), Unique ingre...","[Culinary Arts, Preservation, Cultural History...","Welcome to 'The Magical Marmalade Adventure,' ..."
3,class_023_the_art_of_whimsical_waffle_weaving.pdf,CLASS_1041,The Art of Whimsical Waffle Weaving,Chef Waffleton Crumble,York,Culinary Arts,75.0,[Master the technique of creating intricate wa...,"[Waffle irons of various quirky shapes, Locall...","[Culinary Arts, Creative Cooking, Food Present...",Join us at the School of Dandori for 'The Art ...


In [11]:
# Inspect the scalar columns
df[["class_id", "title", "instructor", "location", "course_type", "cost_gbp"]]

,class_id,title,instructor,location,course_type,cost_gbp
0,CLASS_4033,The Art of Wondrous Waffle Weaving,Chef Waffleby,Harrogate,Culinary Arts,75.0
1,CLASS_8046,Magical Woolly Mammoth Sculpting,Professor Woolly Mammoth,Northumberland,Fiber Arts,75.0
2,CLASS_0425,The Magical Marmalade Adventure,Chef Marmalock,Inverness,Culinary Arts,55.0
3,CLASS_1041,The Art of Whimsical Waffle Weaving,Chef Waffleton Crumble,York,Culinary Arts,75.0


In [12]:
# Inspect the list columns row by row
for _, row in df.iterrows():
    print(f"\n{row['class_id']} — {row['title']}")
    print(f"  Skills:      {row['skills_developed']}")
    print(f"  Objectives:  {row['learning_objectives']}")
    print(f"  Materials:   {row['provided_materials']}")


CLASS_4033 — The Art of Wondrous Waffle Weaving
  Skills:      ['Culinary Arts', 'Baking', 'Creative Cooking', 'Food Presentation', 'Flavour Pairing']
  Objectives:  ['Master the technique of creating intricate waffle patterns.', 'Understand the science behind perfect waffle batter consistency.', 'Explore innovative flavour combinations for both sweet and savoury waffles.', 'Learn artistic presentation techniques for waffle dishes.', 'Develop skills in using waffle irons of various designs.']
  Materials:   ['Professional waffle iron', 'Selection of flours (including gluten-free options)', 'Assorted spices and herbs', 'Fresh fruits and savory toppings', 'Waffle weaving toolkit (spatulas, piping bags, edible glitter)']

CLASS_8046 — Magical Woolly Mammoth Sculpting
  Skills:      ['Needle Felting', 'Sculpting', 'Artistic Expression', 'Creative Crafting', 'Textile Knowledge']
  Objectives:  ['Master the basics of needle felting to create wool sculptures', 'Learn about different types of